# NinaPro DB5 — Data Exploration
**Project:** sEMG Prosthetic FYP | Ian Kiarie | JKUAT  

**How to use this notebook:**  
Run each cell one at a time with **Shift + Enter**.  
Read the comment at the top of each cell before running it.  
Each cell tells you what output to expect.

---

In [ ]:
# ================================================================
# CELL 1 — Imports and path check
# Expected output: '10 subject folders found'
# ================================================================
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from pathlib import Path

NINAPRO_DIR   = Path('../data/ninapro_db5')   # contains S1/ S2/ ... S10/
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(exist_ok=True)

subjects = sorted([d for d in NINAPRO_DIR.iterdir() if d.is_dir()])
print(f'{len(subjects)} subject folders found:')
for s in subjects:
    mats = list(s.rglob('*.mat'))
    print(f'  {s.name}  ->  {len(mats)} .mat files')


In [ ]:
# ================================================================
# CELL 2 — Constants: which channels to use, which gestures to keep
# Expected output: list of 8 gesture classes
# ================================================================
CHANNEL_IDX   = [0, 4, 8, 12]       # 4 of 16 Myo channels
CHANNEL_NAMES = ['FCR','ED','FDS','BR']
FS            = 200                  # DB5 sampling rate Hz

# DB5 gesture label -> our class number
GESTURE_MAP = {0:0, 12:1, 3:2, 11:3, 1:4, 5:5, 6:6, 4:7}

GESTURE_NAMES = {
    0:'Rest', 1:'Open hand', 2:'Power grasp', 3:'Pinch',
    4:'Point', 5:'Wrist flex', 6:'Wrist ext', 7:'Thumbs up'
}

print('8 gesture classes:')
for c, name in GESTURE_NAMES.items():
    db5 = [k for k,v in GESTURE_MAP.items() if v==c][0]
    print(f'  Class {c}: {name:15s}  (DB5 label {db5})')


In [ ]:
# ================================================================
# CELL 3 — Helper functions (no output, just definitions)
# ================================================================

def load_subject(n):
    path = NINAPRO_DIR / f'S{n}' / f's{n}' / f'S{n}_E1_A1.mat'
    m = sio.loadmat(str(path))
    return (m['emg'].astype(np.float32),
            m['restimulus'].flatten().astype(int),
            m['rerepetition'].flatten().astype(int))

def remap_labels(labels):
    mapped = np.full_like(labels, -1)
    for db5, proj in GESTURE_MAP.items():
        mapped[labels == db5] = proj
    return mapped

def extract_features(window):
    feats = []
    for c in range(window.shape[1]):
        x   = window[:, c]
        thr = 0.05 * np.max(np.abs(x)) if np.max(np.abs(x)) > 0 else 1e-6
        dx  = np.diff(x)
        feats.extend([
            np.mean(np.abs(x)),
            np.sqrt(np.mean(x**2)),
            np.sum(np.abs(dx)),
            float(np.sum((x[:-1]*x[1:]<0) & (np.abs(x[:-1]-x[1:])>=thr))),
            float(np.sum((dx[:-1]*dx[1:]<0) & ((np.abs(dx[:-1])+np.abs(dx[1:]))>=thr))),
            np.var(x),
        ])
    return np.array(feats, dtype=np.float32)

def windowed_features(emg4, mapped, window=50, step=10):
    X, y = [], []
    for start in range(0, len(emg4)-window, step):
        lbl = mapped[start:start+window]
        valid = lbl[lbl>=0]
        if len(valid) == 0: continue
        maj = np.argmax(np.bincount(valid+1, minlength=10)) - 1
        if maj < 0: continue
        X.append(extract_features(emg4[start:start+window]))
        y.append(maj)
    return np.array(X), np.array(y)

print('Functions defined OK.')


In [ ]:
# ================================================================
# CELL 4 — Load Subject 1 and inspect
# Expected output: shape (130267, 16), all 8 classes listed
# ================================================================
emg, labels, reps = load_subject(1)
emg4   = emg[:, CHANNEL_IDX]
mapped = remap_labels(labels)

print(f'Full EMG:   {emg.shape}  (time steps x 16 channels)')
print(f'4-ch EMG:   {emg4.shape}  (time steps x 4 channels)')
print(f'Duration:   {emg.shape[0]/FS:.0f} seconds')
print(f'EMG range:  {emg.min():.0f} to {emg.max():.0f}  (8-bit Myo units)')
print()
print('Samples per class:')
for c in range(8):
    n = np.sum(mapped==c)
    print(f'  Class {c} {GESTURE_NAMES[c]:15s}: {n:6d} samples  ({n/FS:.1f} s)')


In [ ]:
# ================================================================
# CELL 5 — Plot: 60-second overview
# Each row = one electrode channel. Shaded = gesture active.
# ================================================================
t    = np.arange(len(emg4)) / FS
mask = t <= 60

fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=True)
fig.suptitle('Real sEMG — Subject 1, First 60 s', fontsize=12)

for i, (ax, name) in enumerate(zip(axes, CHANNEL_NAMES)):
    ax.plot(t[mask], emg4[mask, i], lw=0.4, color=f'C{i}')
    ax.set_ylabel(name, fontsize=9)
    ax.set_ylim(-130, 130)
    ax.grid(alpha=0.2)

for db5_lbl, proj_lbl in GESTURE_MAP.items():
    if db5_lbl == 0: continue
    idx = np.where((labels==db5_lbl) & mask)[0]
    if len(idx) == 0: continue
    for ax in axes:
        ax.axvspan(t[idx[0]], t[idx[-1]], alpha=0.15, color=f'C{proj_lbl}',
                   label=GESTURE_NAMES[proj_lbl])

axes[0].legend(loc='upper right', fontsize=6, ncol=4)
axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.savefig(str(PROCESSED_DIR/'S1_session_overview.png'), dpi=150)
plt.show()


In [ ]:
# ================================================================
# CELL 6 — Plot: one repetition of each gesture side by side
# 8 columns (one per gesture), 4 rows (one per channel)
# ================================================================
fig, axes = plt.subplots(4, 8, figsize=(20, 8), sharey=True)
fig.suptitle('One Rep per Gesture — Subject 1', fontsize=11)

for g in range(8):
    db5 = next(k for k,v in GESTURE_MAP.items() if v==g)
    idx = np.where((labels==db5) & (reps==1))[0]
    if len(idx)==0:
        for ch in range(4): axes[ch,g].set_visible(False)
        continue
    seg  = emg4[idx[0]:idx[-1]+1]
    t_ms = np.arange(len(seg))/FS*1000
    for ch in range(4):
        axes[ch,g].plot(t_ms, seg[:,ch], lw=0.7, color=f'C{ch}')
        axes[ch,g].set_ylim(-130,130)
        axes[ch,g].grid(alpha=0.2)
        axes[ch,g].tick_params(labelsize=6)
        if g==0: axes[ch,g].set_ylabel(CHANNEL_NAMES[ch], fontsize=7)
        if ch==0: axes[ch,g].set_title(GESTURE_NAMES[g], fontsize=8)
        if ch==3: axes[ch,g].set_xlabel('ms', fontsize=6)

plt.tight_layout()
plt.savefig(str(PROCESSED_DIR/'S1_gesture_grid.png'), dpi=150)
plt.show()


In [ ]:
# ================================================================
# CELL 7 — Extract features from Subject 1
# Expected output: X shape (11205, 24), class counts
# ================================================================
X, y = windowed_features(emg4, mapped)

print(f'X shape: {X.shape}  <- (windows x 24 features)')
print(f'y shape: {y.shape}  <- (one label per window)')
print()
print('Windows per class:')
for c in range(8):
    n   = np.sum(y==c)
    bar = '|' * (n//50)
    print(f'  {c} {GESTURE_NAMES[c]:15s}: {n:5d}  {bar}')


In [ ]:
# ================================================================
# CELL 8 — Extract features from ALL 10 subjects (~2 min)
# Watch the progress. Each subject should give ~10-12k windows.
# ================================================================
all_X, all_y = [], []

for n in range(1, 11):
    print(f'Subject {n:2d}...', end=' ', flush=True)
    emg_n, lbl_n, _ = load_subject(n)
    emg4_n  = emg_n[:, CHANNEL_IDX]
    mapped_n = remap_labels(lbl_n)
    Xn, yn   = windowed_features(emg4_n, mapped_n)
    all_X.append(Xn); all_y.append(yn)
    print(f'{len(Xn)} windows')

X_all = np.vstack(all_X)
y_all = np.concatenate(all_y)

print(f'\nFull dataset: {X_all.shape}')
np.save(str(PROCESSED_DIR/'X_all_subjects.npy'), X_all)
np.save(str(PROCESSED_DIR/'y_all_subjects.npy'), y_all)
print('Saved X_all_subjects.npy and y_all_subjects.npy')


In [ ]:
# ================================================================
# CELL 9 — Balance classes (fix Rest dominance) and save
# ================================================================
from sklearn.utils import shuffle

target = int(np.mean([np.sum(y_all==c) for c in range(1,8)]))
print(f'Target count per class: {target}')

rest_idx    = np.where(y_all==0)[0]
gesture_idx = np.where(y_all>0)[0]
keep_rest   = np.random.choice(rest_idx, size=target, replace=False)
idx_bal     = np.concatenate([keep_rest, gesture_idx])

X_bal, y_bal = shuffle(X_all[idx_bal], y_all[idx_bal], random_state=42)

print(f'Balanced dataset: {X_bal.shape}')
print('Class counts after balancing:')
for c in range(8):
    print(f'  {c} {GESTURE_NAMES[c]:15s}: {np.sum(y_bal==c)}')

np.save(str(PROCESSED_DIR/'X_balanced.npy'), X_bal)
np.save(str(PROCESSED_DIR/'y_balanced.npy'), y_bal)
print('\nSaved X_balanced.npy and y_balanced.npy')
print('Ready for training!')
